## Model development using Efficient-B0 with FastAI framework

This notebook presents a robust image classification pipeline for paddy leaf disease detection using a fine-tuned EfficientNet-B0 model with the FastAI framework and Test-Time Augmentation (TTA) for ensembling. The workflow includes data loading, preprocessing, augmentation, model configuration, and performance evaluation. Leveraging extensive augmentations and W&B for experiment tracking, the model achieved an impressive 98.35% accuracy on the test set. Evaluation using a classification report and confusion matrix confirms high precision and recall across all classes, validating the model’s effectiveness for real-world deployment in agricultural disease monitoring.

<!-- ####  a. Import dataset

1. Download the Paddy Doctor split balanced dataset file ``paddy-doctor-diseases-small-400-split.zip`` from [this link](https://ieee-dataport.org/documents/paddy-doctor-visual-image-dataset-automated-paddy-disease-classification-and-benchmarking)
2. Create a new folder named ``paddy-doctor`` in your Google Drive's root directory and upload the ``paddy-doctor-diseases-small-400-split.zip`` file there. -->

Environment Setup and Data Loading for FastAI-based Training
-------

In [ ]:
# Import glob for file operations
import glob

# Try importing fastkaggle, install if not already available
try:
    import fastkaggle
except ModuleNotFoundError:
    !pip install -Uq fastkaggle

# Import key modules from fastkaggle and fastai
from fastkaggle import *
from fastai.vision.all import *

# Set random seed for reproducibility
set_seed(42)

# Define path to the competition dataset (local or Kaggle format)
competition = '/kaggle/input/paddy-doctor-disease/Augmented and split - 26000 augmented images split into train (80) sets'

# Use fastkaggle helper to set up the dataset and environment
# Also installs fastai and timm (PyTorch image models)
path = setup_comp(competition, install='fastai "timm>=0.6.2.dev0"')
print(path)

# Define path and load all image files for training
train_path = path / 'train'
train_files = get_image_files(train_path)

# Define path and load all image files for testing (sorted for consistency)
test_path = path / 'test'
test_files = get_image_files(test_path).sorted()

# Load training labels from metadata CSV file
train_df = pd.read_csv(path / 'metadata-train.csv')
print(train_df.shape)

# Display class distribution in the training dataset
train_df.label.value_counts()

DataBlock Definition and DataLoaders Creation with Augmentations
----------

In [ ]:
# Define a DataBlock for image classification using FastAI
dblock = DataBlock(
    blocks=(ImageBlock, CategoryBlock),       # Define the input (images) and target (categories) types
    get_items=get_image_files,                # Function to retrieve image file paths
    get_y=parent_label,                       # Use the parent folder name as the label
    splitter=RandomSplitter(valid_pct=0.2, seed=42),  # Randomly split 20% of data for validation
    item_tfms=Resize(480, method='squish'),   # Resize all images to 480x480 using 'squish' to fit without padding
    batch_tfms=aug_transforms(                # Apply data augmentations on batches
        size=224,                             # Final image size after crop
        min_scale=0.75,                       # Minimum scale for cropping
        max_rotate=20,                        # Allow rotation up to 20 degrees
        max_zoom=1.2,                         # Allow zoom-in up to 1.2x
        max_warp=0.2,                         # Apply perspective warping
        p_affine=0.75,                        # 75% chance of applying affine transforms
        p_lighting=0.75                       # 75% chance of applying lighting transforms
    ) + [RandomErasing(p=0.5, max_count=1)]   # Randomly erase one region in 50% of the images for robustness
)

# Create the DataLoaders (train and valid loaders)
dls = dblock.dataloaders(train_path)

Quick DataLoaders Creation Using ImageDataLoaders.from_folder
----------

In [ ]:
# Create ImageDataLoaders directly from folders using FastAI's high-level API
dls = ImageDataLoaders.from_folder(
    train_path,                       # Root directory with class folders
    valid_pct=0.2,                    # Use 20% of data for validation
    seed=42,                          # Seed for reproducible split
    item_tfms=Resize(480, method='squish'),  # Resize all images to 480x480 using 'squish' (no padding)
    batch_tfms=aug_transforms(        # Apply default data augmentations on batches
        size=224,                     # Crop to 224x224 after resizing
        min_scale=0.75               # Allow scaling down to 75% before cropping
    )
)

Weights & Biases (wandb) Initialization for EfficientNet Training Run
--------------

In [ ]:
# Import Weights & Biases for experiment tracking
import wandb

# Log in to Weights & Biases using an API key
wandb.login(key="")

# Initialize a new W&B run for the EfficientNet experiment
run = wandb.init(
    project="project-ablations",     # Project name on wandb dashboard
    name="efficientNet",             # Run name to identify this experiment
    config={                         # Hyperparameter configuration
        "epochs": 20,                 # Number of training epochs
        "base_lr": 0.005,             # Base learning rate
        "weight_decay": 0.01,         # L2 regularization value
        "architecture": "EfficientNet"  # Model architecture name
    },
    reinit=True                      # Allow reinitializing this run if rerun in the same process
)

Creating and Configuring EfficientNet Learner with FastAI and W&B
---------

In [ ]:
# Import model creation utility and FastAI components
from timm import create_model
from fastai.vision.all import vision_learner, error_rate
from fastai.callback.wandb import WandbCallback

# -----------------------------
# EfficientNet Learner
# -----------------------------

# Define a wrapper to create an EfficientNet-B0 model using the timm library
def efficientnet_model(**kwargs):
    kwargs.pop('pretrained', None)  # Remove any externally passed 'pretrained' to avoid conflict
    return create_model("tf_efficientnet_b0_ns", pretrained=True, **kwargs)

# Create a FastAI vision learner using the EfficientNet backbone
learn_eff = vision_learner(
    dls,                          # DataLoaders object with training/validation sets
    efficientnet_model,          # Custom EfficientNet model function
    metrics=error_rate,          # Evaluation metric to monitor during training
    cbs=[
        MixUp(),                 # Data augmentation callback (MixUp for regularization)
        SaveModelCallback(       # Save best model based on error rate
            fname='eff_best_model',
            monitor='error_rate'
        ),
        WandbCallback(log_model=True)  # Log metrics and model checkpoints to wandb
    ]
).to_fp16()                       # Convert model to 16-bit floating point (mixed precision training)

# Set directory where model checkpoints will be saved
learn_eff.model_dir = "/kaggle/working/models"

Fine-Tuning EfficientNet Model
---------

In [ ]:
# Fine-tune the EfficientNet model for 40 epochs
# using a base learning rate of 0.005 and weight decay of 0.01
learn_eff.fine_tune(40, base_lr=0.005, wd=0.01)

Exporting Trained Model and Logging to Weights & Biases
------------

In [ ]:
# Export the trained EfficientNet model as a pickle file for inference or deployment
learn_eff.export("/kaggle/working/eff_final_model.pkl")

# Log the exported model file to Weights & Biases for record-keeping and sharing
wandb.save("eff_final_model.pkl")

Ensemble Predictions Using Test-Time Augmentation (TTA)
-----------

In [ ]:
# Perform Test-Time Augmentation (TTA) on the validation set
# This returns averaged predictions over several augmented versions of each image
probs_eff, target = learn_eff.tta(dl=dls.valid)

# Print the validation error rate using the TTA-averaged predictions
print("EfficientNet Error Rate (Validation):", error_rate(probs_eff, target))

Test Set Predictions with Ensemble Using TTA
----------

In [ ]:
# Define the test directory and retrieve all test image files (sorted for order consistency)
test_path = path / 'test'
test_files = get_image_files(test_path).sorted()

# Get the true class names from parent directories (if available)
test_classes = [f.parent.name for f in test_files]

# Perform Test-Time Augmentation (TTA) and predict on test set
# Returns class probabilities for each test image
probs_eff_test, _ = learn_eff.tta(dl=dls.test_dl(test_files))

# Get the predicted class indices by selecting the class with highest probability
preds = probs_eff_test.argmax(dim=1)

# Convert predicted indices to actual class names using vocab mapping
pred_classes = dls.vocab[preds]


**Results**

Test Set Evaluation: Accuracy and Classification Report
--------

In [ ]:
# Import evaluation metrics and visualization tools
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns

# Generate a detailed classification report
cls_report = classification_report(
    test_classes,        # True class labels (from folder names)
    pred_classes,        # Predicted class labels
    digits=5             # Decimal precision for scores
)
print(cls_report)

# Calculate and print the overall accuracy on the test set
acc = accuracy_score(test_classes, pred_classes)
print(f"EfficientNet Accuracy on Test Set: {acc:.5f}")

Confusion Matrix Heatmap for Test Set Predictions
------------

In [ ]:
# Import seaborn for heatmap and sklearn for confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Define a function to plot a confusion matrix heatmap
def plot_heatmap(y_true, y_pred, class_names, ax, title):
    cm = confusion_matrix(y_true, y_pred)  # Compute confusion matrix
    sns.heatmap(
        cm, 
        annot=True,                          # Annotate cells with counts
        square=True,                         # Square-shaped cells
        xticklabels=class_names,             # X-axis class labels
        yticklabels=class_names,             # Y-axis class labels
        fmt='d',                             # Format as integers
        cmap=plt.cm.Blues,                   # Use blue color map
        cbar=False,                          # Hide color bar
        ax=ax                                # Plot on provided axis
    )
    # Set axis labels and tick formatting
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=12, rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_xlabel('Predicted Label', fontsize=12)

# Create a matplotlib figure and axis
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

# Plot confusion matrix using the test predictions
plot_heatmap(test_classes, pred_classes, dls.vocab, ax, title="Ensembled EfficientNet + ResNet50")

# Display the plot
plt.show()

Saving Prediction Results to CSV
---------

In [ ]:
# Create a DataFrame to store true and predicted labels for the test set
res = pd.DataFrame({
    "y_true": test_classes,      # True class labels (from test directory)
    "y_pred": pred_classes       # Predicted class labels (from model)
})

# Save the results to a CSV file for further analysis or review
res.to_csv('result.csv', index=False)

# Display the result DataFrame
res